In [3]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ======================
# LOAD
# ======================
df = pd.read_csv('../data/full_combinations_flights.csv')
df.columns = df.columns.str.strip().str.lower()
df = df.dropna()

# ======================
# FIX TOTAL STOPS
# ======================
def get_column(df, names):
    for n in names:
        if n in df.columns:
            return df[n]
    raise Exception(f"Missing column: {names}")

df["total_stops"] = get_column(df, ["total_stops", "totalstops", "total stops"])

# ======================
# DATETIME FEATURES
# ======================
df['departuredatetime'] = pd.to_datetime(df['departuredatetime'])

df['DepartureHour'] = df['departuredatetime'].dt.hour
df['DepartureDay'] = df['departuredatetime'].dt.day
df['DepartureMonth'] = df['departuredatetime'].dt.month
df['DepartureWeekday'] = df['departuredatetime'].dt.weekday
df["IsWeekend"] = df["DepartureWeekday"].isin([5,6]).astype(int)

today = df['departuredatetime'].min()
df["DaysUntilDeparture"] = (df['departuredatetime'] - today).dt.days

# ======================
# ROUTE STATS
# ======================
stats = df.groupby(['source', 'destination']).agg(
    price_min=('price', 'min'),
    price_max=('price', 'max'),
    price_avg=('price', 'mean')
).reset_index()

df = df.merge(stats, on=['source', 'destination'], how='left')

# ======================
# LABEL
# ======================
df['Label'] = np.where(df['price'] <= df['price_avg'] * 0.9, 1, 0)

# ======================
# FEATURES
# ======================
categorical = ["airline", "source", "destination"]
numerical = [
    "total_stops",
    "durationminutes",
    "DepartureHour",
    "DepartureDay",
    "DepartureMonth",
    "DepartureWeekday",
    "IsWeekend",
    "DaysUntilDeparture",
    "price_avg",
    "price_min",
    "price_max"
]

features = categorical + numerical

X = df[features]

# ======================
# PREPROCESSOR
# ======================
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", MinMaxScaler(), numerical)
])

# ======================
# REGRESSOR PIPELINE
# ======================
reg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, max_depth=14))
])

# ======================
# CLASSIFIER PIPELINE
# ======================
clf_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200))
])

# ======================
# TRAIN
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, df["price"], test_size=0.2, random_state=42
)

reg_pipeline.fit(X_train, y_train)
pred = reg_pipeline.predict(X_test)

print("REGRESSION:")
print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

# ======================
# CLASSIFIER
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, df["Label"], test_size=0.2, random_state=42
)

clf_pipeline.fit(X_train, y_train)
pred = clf_pipeline.predict(X_test)

print("\nCLASSIFIER:")
print("Accuracy:", accuracy_score(y_test, pred))

# ======================
# SAVE
# ======================
os.makedirs("../models", exist_ok=True)

joblib.dump(reg_pipeline, "../models/reg_pipeline.pkl")
joblib.dump(clf_pipeline, "../models/clf_pipeline.pkl")
joblib.dump(stats, "../models/route_stats.pkl")

print("✅ Pipeline models saved")

REGRESSION:
MAE: 1193.5118745825368
R2: 0.8396511895957594

CLASSIFIER:
Accuracy: 0.8590760615958936
✅ Pipeline models saved
